# Labour Code Guide
- Ask a question about federal labour standards (Canada Labour Code, Part III). The system finds the most relevant sections of the law and Claude answers using only those sections, citing the section numbers so you can check the source.

## Data Trust

- **Source:** Canada Labour Code (R.S.C. 1985, c. L-2)
- **License:** Reproduction of Federal Law Order (SI/97-5): can be reproduced without permission; this is not an official version.
- **Scope:** Part III covers federally regulated workplaces only (banks, airlines, telecoms…). Most jobs fall under provincial law instead.
- As current as December 29, 2025 (last amended December 12, 2025)


## Create table from XML

In [1]:
# Prediction: There will be over 100 rows
import xml.etree.ElementTree as ET
import pandas as pd

root = ET.parse("data/raw/L-2.xml").getroot()
body = root.find("Body")

rows, part, division = [], None, None
for el in body:                      # Body is a flat run of Heading and Section elements
    if el.tag == "Heading":
        title = " ".join(el.find("TitleText").itertext()).strip()
        if el.get("level") == "1":   # a Part
            part, division = title, None
        elif el.get("level") == "2": # a Division inside the Part
            division = title
    elif el.tag == "Section" and part == "Standard Hours, Wages, Vacations and Holidays":
        for parent in el.iter():     # drop amendment-history notes; they aren't the law's text
            for note in parent.findall("HistoricalNote"):
                parent.remove(note)
        rows.append({
            "section": el.findtext("Label"),
            "division": division,
            "title": (el.findtext("MarginalNote") or "").strip(),
            "text": " ".join(" ".join(el.itertext()).split()),
        })

sections = pd.DataFrame(rows)
len(sections)

217

## Remove repealed sections

In [2]:
repealed = sections["text"].str.contains("[Repealed", regex=False) & (sections["text"].str.len() < 120)
sections = sections[~repealed]
sections.to_csv("data/sections.csv", index=False)
len(sections)

205

## Check Data

In [5]:
sections[sections["section"] == "184"].iloc[0]["text"]     # does it read correctly?

'Annual vacation with pay 184 Except as otherwise provided by or under this Division, in respect of every year of employment by an employer, every employee is entitled to and shall be granted a vacation with vacation pay of (a) at least two weeks if they have completed at least one year of employment; (b) at least three weeks if they have completed at least five consecutive years of employment with the same employer; and (c) at least four weeks if they have completed at least 10 consecutive years of employment with the same employer.'

In [6]:
sections["text"].str.len().describe()                       # how long are the chunks?

count     205.000000
mean     1151.160976
std      1075.136744
min        75.000000
25%       408.000000
50%       831.000000
75%      1590.000000
max      6365.000000
Name: text, dtype: float64

In [4]:
sections["division"].value_counts()                         # what topics are covered?

division
Administration and General                                   46
Maternity-related Reassignment and Leave and Other Leaves    31
Group Termination of Employment                              19
Hours of Work                                                16
Annual Vacations                                             12
General Holidays                                             11
Leave of Absence for Members of the Reserve Force            11
Unjust Dismissal                                              9
Bereavement Leave                                             7
Individual Terminations of Employment                         6
Complaints Relating to Reprisals                              6
Application                                                   5
Minimum Wage and Age of Employment                            4
Genetic Testing                                               4
Breaks for Medical Reasons or Nursing                         3
Severance Pay                  

In [7]:
sections.loc[sections["text"].str.len().idxmax(), ["section", "title"]]

section                     239
title      Entitlement to leave
Name: 129, dtype: str

## Chunking Decision
- One law section = one chunk, so each answer can cite a section number.
- Most sections are short (median about 831 characters); a few are long (longest: s.239 )
- If retrieval struggles with the long ones later, I can split them further.

## Test Set
- 20 plain language questions across 9 topics (most with 2 phrasings), plus 3 trap questions Part III can't answer. Labels verified against section text.
- Coverage: Hours of work is the most tested (6 Questions); Maternity has no questions yet, so results say nothing about maternity leave and reassignment. That's a gap to fill when the test set grows.
- **Label change (after review):** "How do I file a complaint with the Ministry of Labour?" was
  originally a trap (`none`). On review, it is partly answerable: s. 251.01 sets out the federal
  complaint process, so it was relabelled to 251.01. Result: 18 answerable questions, 2 traps.
- **Added after launch:** "How many overtimes can I work?" (s. 171), a real failure found in live app. Not yet scored; it will be included in next evaluation round.

In [69]:
questions = pd.read_csv("data/questions.csv", dtype={"expected_section": str})
check = questions.merge(sections, left_on="expected_section", right_on="section", how="left")
check[["question", "expected_section", "title"]]

,question,expected_section,title
0,How much vacation am I entitled to?,184,Annual vacation with pay
1,How many weeks of vacation do I get after work...,184,Annual vacation with pay
2,How is my vacation pay calculated?,184.01,Calculation of vacation pay
3,How much vacation pay am I owed if I am laid off?,188,Termination of employment during year
4,What is the most hours I can be made to work i...,171,Maximum hours of work
5,How many hours are in a standard work week?,169,Standard hours of work
6,How many hours of rest am I entitled to betwee...,169.2,Rest period
7,How much time off do I get between two shifts?,169.2,Rest period
8,"My employer is refusing to pay me overtime, wh...",174,Overtime pay or time off
9,What rate am I paid for overtime?,174,Overtime pay or time off


In [11]:
check["division"].value_counts(dropna=False)

division
Hours of Work                            6
Annual Vacations                         4
Individual Terminations of Employment    3
NaN                                      3
Severance Pay                            2
Unjust Dismissal                         1
Administration and General               1
Name: count, dtype: int64

## Retrieval

In [70]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

STOPWORDS = {"how", "much", "many", "what", "when", "can", "do", "i", "am", "is", "are", "the",
             "a", "an", "to", "of", "my", "me", "if", "in", "for", "on", "be", "have", "with",
             "after", "or", "does", "should", "by", "from", "it", "get", "that", "this", "and"}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())   # lowercase words and numbers only
    return [w for w in words if w not in STOPWORDS]

corpus_tokens = [tokenize(t) for t in sections["text"]]
answerable = questions[questions["expected_section"] != "none"]

## Method A: Find the Baseline

In [25]:
def search_count(question, k =3):
    q = tokenize(question)
    scores = np.array([sum(doc.count(w) for w in q) for doc in corpus_tokens])
    return sections["section"].iloc[np.argsort(-scores)[:k]].tolist()

search_count("How much time off do I get between two shifts?")

['174', '178', '238.1']

## Method B: BM25 

In [24]:
bm25 = BM25Okapi(corpus_tokens)

def search_bm25(question, k=3):
    scores = bm25.get_scores(tokenize(question))
    return sections["section"].iloc[np.argsort(-scores)[:k]].tolist()

search_bm25("How much time off do I get between two shifts?")

['169.2', '238.1', '174']

In [71]:
# Prediction: Word count hit@3 = about 40%, BM25 hit@3 = about 60% → changed to 75% before running. Actual see below: 

def hit_rate(search, k):
    hits = [row["expected_section"] in search(row["question"], k) for _, row in answerable.iterrows()]
    return np.mean(hits)

pd.DataFrame({"word count": [hit_rate(search_count, k) for k in (1, 3, 5)],
              "BM25": [hit_rate(search_bm25, k) for k in (1, 3, 5)]},
             index=["hit@1", "hit@3", "hit@5"]).round(2)

,word count,BM25
hit@1,0.17,0.44
hit@3,0.39,0.61
hit@5,0.44,0.83


## Read the misses

In [72]:
misses = answerable.assign(top3=[search_bm25(q, 3) for q in answerable["question"]])
misses[[e not in t for e, t in zip(misses["expected_section"], misses["top3"])]][["question", "expected_section", "top3"]]

,question,expected_section,top3
0,How much vacation am I entitled to?,184,"[185, 184.01, 187]"
2,How is my vacation pay calculated?,184.01,"[185, 191, 186]"
3,How much vacation pay am I owed if I am laid off?,188,"[185, 251.001, 242]"
13,How do I calculate severance pay if I have bee...,235,"[242, 236, 174]"
14,Am I entitled to severance pay if I am laid of...,235,"[189, 174, 236]"
15,I think I was fired unfairly. Can I file a com...,240,"[251.01, 251.1, 182]"
16,How long after being laid off do I have to fil...,251.01,"[182, 242, 246.1]"


In [32]:
# Why does s. 185 win every vacation question? (dense use of "vacation")
sections[sections["section"] == "185"].iloc[0]["text"]


'Granting vacation with pay 185 The employer of an employee who under this Division has become entitled to a vacation with vacation pay (a) shall grant to the employee the vacation to which the employee is entitled, which shall begin not later than ten months immediately following the completion of the year of employment for which the employee became entitled to the vacation; and (b) shall, at any time that is prescribed by the regulations, pay to the employee (i) if the vacation is taken in one period, the vacation pay to which the employee is entitled in respect of that vacation, or (ii) if the vacation is taken in more than one period, for each period, the proportion of the vacation pay that the vacation taken is of the annual vacation to which the employee is entitled.'

In [33]:
# "laid off" appears once in Part III, and not in the severance section
sections[sections["text"].str.contains("laid off", case=False)][["section", "title"]]

,section,title
138,242,


In [35]:
sections[sections["section"] == "235"].iloc[0]["text"]

'Minimum rate 235 (1) An employer who terminates the employment of an employee who has completed twelve consecutive months of continuous employment by the employer shall, except where the termination is by way of dismissal for just cause, pay to the employee the greater of (a) two days wages at the employee’s regular rate of wages for his regular hours of work in respect of each completed year of employment that is within the term of the employee’s continuous employment by the employer, and (b) five days wages at the employee’s regular rate of wages for his regular hours of work. Clarification (1.1) The employer’s obligation to pay and the employee’s right to receive the amount under subsection (1) apply whether or not the employee has a right to avail themselves of any procedure for redress under this Part, including under subsection 240(1), with respect to the termination of their employment. Circumstances deemed to be termination and deemed not to be termination (2) For the purposes

In [31]:
sections[sections["section"] == "240"].iloc[0]["text"]

'Complaint 240 (1) Subject to subsections (2) and 242(3.1), a person who has been dismissed and considers the dismissal to be unjust may make a complaint in writing to the Head if the employee (a) has completed 12 consecutive months of continuous employment by an employer; and (b) is not a member of a group of employees subject to a collective agreement. For greater certainty (1.01) For greater certainty, the fact that an employer complies with Divisions X and XI does not (a) affect the employee’s rights under this Division; or (b) prevent the Board from deciding under paragraph 242(3)(a) that the dismissal was unjust or from making any order under subsection 242(4), including a reinstatement order. Limitation (1.1) A person shall not make a complaint under subsection (1) if they have made a complaint that is based on substantially the same facts under either subsection 246.1(1) or 247.99(1), unless that complaint has been withdrawn. Time for making complaint (2) Subject to subsection 

## Choosing k

In [73]:
pd.Series({k: hit_rate(search_bm25, k) for k in range(1, 11)}).round(2)

1     0.44
2     0.61
3     0.61
4     0.72
5     0.83
6     0.83
7     0.83
8     0.83
9     0.89
10    0.89
dtype: float64

## Retrieval findings
- **Dense keyword sections win:** s. 185 uses "vacation" 12 times, so it tops every vacation
  question even though it covers *when* vacation is granted, not *how much*.
- **Vocabulary mismatch:** workers' words vs the law's words —
  "laid off" → "terminates the employment" (s. 235); "fired" → "dismissed"; "unfairly" → "unjust" (s. 240).
  BM25 matches exact words only, so it can't connect them.
- **Chose k=5:** it's where hit@k plateaus (0.83). k=9 adds one question (0.89) but nearly doubles the context, and one question is within the noise at n=18. Next step: compare answer quality at k=5 vs k=9 once Claude generates answers.
- **Caveat:** Sample size is 18 questions, so each one moves hit@k by about 5.5 points. Treat the small differences as noise

## Can retrieval detect unanswerable questions?

In [74]:
# Can BM25's top score alone tell trap questions apart from answerable ones?
top_scores = questions.assign(
    top_score=[bm25.get_scores(tokenize(q)).max() for q in questions["question"]],
    trap=questions["expected_section"] == "none",
)
top_scores.sort_values("top_score")[["top_score", "trap", "expected_section", "question"]].round(1)


,top_score,trap,expected_section,question
11,3.7,False,230,"If I get let go, how many weeks of warning do ..."
10,5.5,False,230,How much notice am I entitled to if my employe...
18,6.3,False,251.01,How do I file a complaint with the Ministry of...
0,6.5,False,184,How much vacation am I entitled to?
19,6.9,True,none,How do I negotiate a raise or promotion with m...
2,7.1,False,184.01,How is my vacation pay calculated?
3,7.1,False,188,How much vacation pay am I owed if I am laid off?
15,7.6,False,240,I think I was fired unfairly. Can I file a com...
17,7.7,True,none,How much notice do I need to give my employer ...
13,9.0,False,235,How do I calculate severance pay if I have bee...


In [40]:
# Summarize the top scores by trap vs. answerable questions
top_scores.groupby("trap")["top_score"].agg(["min", "max"]).round(1)

,min,max
trap,,
False,3.7,21.7
True,6.3,7.7


## Conclusion
- Trap questions still get BM25 scores (6.3–7.7), and that range sits inside the real
  questions' range (3.7–21.7), so no score cut-off can separate them.
- The two lowest scores are real questions ("let go", "warning" → s. 230), so a cut-off
  would also block answerable questions.
- Retrieval can't decide "this isn't covered". The answer step has to: Claude must read
  the retrieved sections and say so when they don't answer the question.

## Answer Generation:

In [87]:
from prompt import SYSTEM

In [88]:
import json
from pathlib import Path
import anthropic

client = anthropic.Anthropic()

ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "covered": {"type": "boolean"},
        "answer": {"type": "string"},
        "cited_sections": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["covered", "answer", "cited_sections"],
    "additionalProperties": False,
}

def build_context(section_ids):
    chosen = sections.set_index("section").loc[section_ids]
    return "\n\n".join(f"[s. {sid}] {row['title']}\n{row['text']}" for sid, row in chosen.iterrows())

def answer(question, k=5, system=SYSTEM):
    retrieved = search_bm25(question, k)
    response = client.beta.messages.create(
        model="claude-opus-5-5",
        max_tokens=4096,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        output_config={"effort": "medium",
                       "format": {"type": "json_schema", "schema": ANSWER_SCHEMA}},
        system=system,
        messages=[{"role": "user",
                   "content": f"Sections of the Canada Labour Code, Part III:\n\n{build_context(retrieved)}"
                              f"\n\nQuestion: {question}"}],
    )
    if response.stop_reason == "refusal":
        return {"covered": None, "answer": "", "cited_sections": [], "retrieved": retrieved}
    text = next(block.text for block in response.content if block.type == "text")
    return json.loads(text) | {"retrieved": retrieved}

In [57]:
answer("How much time off do I get between two shifts?")

{'covered': True,
 'answer': "Under the federal rules, you are entitled to a rest period of at least 8 hours in a row between the end of one shift and the start of your next one, and your employer must give it to you (s. 169.2).\n\nThere is one exception. Your employer can ask you to work extra hours that cut your rest period below 8 hours, but only when you are needed to deal with a situation the employer could not reasonably have seen coming, and that poses (or could reasonably be expected to pose) an imminent or serious threat of:\n- harm to someone's life, health or safety;\n- damage to or loss of property; or\n- serious disruption to the normal operation of the employer's workplace (s. 169.2).\n\nOutside of these urgent, unforeseeable situations, your employer must give you at least 8 hours in a row off between shifts.\n\nThese rules apply only to federally regulated workplaces, such as banks, airlines, telecommunications and interprovincial transportation. If you work for a provi

In [58]:
answer("If I get let go, how many weeks of warning do I get?")

{'covered': True,
 'answer': 'Under the federal rules, if your employer ends your job, they must either give you written notice ahead of time, pay you your regular wages instead of notice (called "pay in lieu of notice"), or a mix of both that adds up to the required number of weeks (s. 230(1)). How many weeks you get depends on how long you have worked for that employer without a break (s. 230(1.1)):\n\n- At least 3 months: 2 weeks\n- At least 3 years: 3 weeks\n- At least 4 years: 4 weeks\n- At least 5 years: 5 weeks\n- At least 6 years: 6 weeks\n- At least 7 years: 7 weeks\n- At least 8 years or more: 8 weeks\n\nOther points that may help:\n- Being laid off is generally treated the same as being let go for these rules, unless a regulation says otherwise (s. 230(3)).\n- You have the right to this notice or pay even if you can also file a complaint about the firing (s. 230(1.01)).\n- Your employer must also give you a written statement listing your vacation pay, wages, severance pay an

In [59]:
answer("How do I negotiate a raise or promotion with my employer?")

{'covered': False,
 'answer': 'The sections of the Canada Labour Code, Part III that were provided do not explain how to negotiate a raise or a promotion with your employer. The law does not set out a process or tips for asking for more pay or a better position.\n\nThe provided sections only touch on related topics. For example, they cover your right to be told in writing about promotion or training opportunities while you are on certain types of leave, if you ask in writing. They also protect you from being punished, or passed over for a promotion, because you made a complaint or used your rights under the Code. They do not cover how to negotiate pay or a promotion.\n\nFor help, you could:\n- Check your employment contract or workplace policies.\n- Check your collective agreement and talk to your union representative, if you are unionized.\n- Contact the federal Labour Program (Employment and Social Development Canada) for general information.\n\nThis information is for general purpos

## Run All questions

In [76]:
def run_all(k, path):
    path = Path(path)
    if not path.exists():
        rows = [answer(q, k) for q in questions["question"]]
        out = questions.assign(**pd.DataFrame(rows))
        out["cited_sections"] = out["cited_sections"].map(";".join)   # list → "184;184.01"
        out["retrieved"] = out["retrieved"].map(";".join)
        out.to_csv(path, index=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

# Prediction: Claude will answer reals questions at 70% in k=5 and 80% in k=9
answers_k5 = run_all(5, "data/answers_k5.csv").assign(expected_section=questions["expected_section"])
answers_k9 = run_all(9, "data/answers_k9.csv").assign(expected_section=questions["expected_section"])

## RAG scorecard

In [77]:
def rag_scorecard(r):
    r = r.assign(covered=r["covered"] == "True",
                 cited=r["cited_sections"].str.split(";"),
                 retrieved=r["retrieved"].str.split(";"))
    real = r[r["expected_section"] != "none"]
    traps = r[r["expected_section"] == "none"]
    return {
        "retrieval hit (in context)": np.mean([e in t for e, t in zip(real["expected_section"], real["retrieved"])]),
        "cites the right section": np.mean([e in c for e, c in zip(real["expected_section"], real["cited"])]),
        "answers real questions": real["covered"].mean(),
        "flags traps as not covered": (~traps["covered"]).mean(),
    }

pd.DataFrame({"k=5": rag_scorecard(answers_k5), "k=9": rag_scorecard(answers_k9)}).round(2)

,k=5,k=9
retrieval hit (in context),0.83,0.89
cites the right section,0.83,0.89
answers real questions,0.83,0.94
flags traps as not covered,1.00,1.00


## Generation misses

In [82]:
r = answers_k5.assign(cited=answers_k5["cited_sections"].str.split(";"),
                      retrieved=answers_k5["retrieved"].str.split(";"))
real = r[r["expected_section"] != "none"]

generation_misses = real[[ (e in t) and (e not in c) for e, t, c in zip(real["expected_section"], real["retrieved"], real["cited"]) ]]
generation_misses[["question", "expected_section", "cited_sections"]]


,question,expected_section,cited_sections


In [83]:
r = answers_k9.assign(cited=answers_k5["cited_sections"].str.split(";"),
                      retrieved=answers_k5["retrieved"].str.split(";"))
real = r[r["expected_section"] != "none"]

generation_misses = real[[ (e in t) and (e not in c) for e, t, c in zip(real["expected_section"], real["retrieved"], real["cited"]) ]]
generation_misses[["question", "expected_section", "cited_sections"]]


,question,expected_section,cited_sections


There were no generational misses for either k, which says whenever the right section was retrieved Claude in fact cited it. The remaining errors are retrieval misses.

## Read the answers

In [78]:
answers_k5[["question", "covered", "cited_sections", "answer"]]

,question,covered,cited_sections,answer
0,How much vacation am I entitled to?,True,184;184.01;185;187;187.1,"Under the federal rules (Canada Labour Code, P..."
1,How many weeks of vacation do I get after work...,True,184;184.01,"Under the federal rules, once you have worked ..."
2,How is my vacation pay calculated?,True,184.01;185;186;187,Under the Canada Labour Code (which applies to...
3,How much vacation pay am I owed if I am laid off?,False,,"The sections of the Canada Labour Code, Part I..."
4,What is the most hours I can be made to work i...,True,171;169;172;173,"Under the federal rules, the general limit is ..."
5,How many hours are in a standard work week?,True,169;170;171;173;166,Under the Canada Labour Code (which applies to...
6,How many hours of rest am I entitled to betwee...,True,169.2;173;166,"Under the Canada Labour Code, you are entitled..."
7,How much time off do I get between two shifts?,True,169.2,"Under the Canada Labour Code, you have the rig..."
8,"My employer is refusing to pay me overtime, wh...",True,174;258;255,This is for informational purposes only and is...
9,What rate am I paid for overtime?,True,174;197,Under the Canada Labour Code (federally regula...


In [79]:
row = answers_k5[answers_k5["question"].str.contains("Ministry")].iloc[0]
print(row["question"], "\n")
print(row["answer"])

How do I file a complaint with the Ministry of Labour? 

Under the Canada Labour Code, you don't file a complaint with a 'Ministry of Labour.' Instead, you send it to an official the law calls 'the Head.' Note that 'Ministry of Labour' usually refers to a provincial government office. The Canada Labour Code only covers federally regulated workplaces, such as banks, airlines, telecommunications and interprovincial transportation. If your workplace is provincially regulated, these rules do not apply, and you would need to check your province's own labour laws.

Here is what the sections provided say about filing a complaint under the federal rules:

1. **Who can complain and about what:** Any employee can complain if they believe their employer has broken a rule in Part III of the Code, its regulations, or an order (s. 251.01(1)).

2. **How:** The complaint must be made **in writing** to the Head (s. 251.01(1)).

3. **Deadline:** You generally have **six months** to file (s. 251.01(2)):


In [80]:
row = answers_k9[answers_k9["question"].str.contains("laid off") & answers_k9["question"].str.contains("vacation")].iloc[0]
print(row["question"], "\n")
print(row["answer"])

How much vacation pay am I owed if I am laid off? 

The sections provided explain how vacation pay is calculated, but they do not include the specific rules on paying out vacation pay when someone is laid off or their job ends. Here is what the provided sections do say.

**How much vacation pay you earn (s. 184.01)**
Vacation pay is a percentage of the wages you earned during the year of employment that the vacation is for:
- **4% of your wages** as the basic amount.
- **6% of your wages** if you have worked at least 5 years in a row for the same employer.
- **8% of your wages** if you have worked at least 10 years in a row for the same employer.

For example, if you earned $50,000 in that year and have worked there for 6 years in a row, your vacation pay would be 6% of $50,000, or $3,000.

**Vacation pay counts as wages (s. 186)**
The law treats vacation pay as wages for all purposes. This means it is money your employer owes you, just like your regular pay.

**Layoffs and timing of p

In [81]:
row = answers_k9[answers_k9["question"].str.contains("raise")].iloc[0]
print(row["question"], "\n")
print(row["answer"])

How do I negotiate a raise or promotion with my employer? 

The sections of the Canada Labour Code, Part III provided here don't cover how to negotiate a raise or a promotion with your employer, so I can't give you guidance on that from the law.

A few related points in these sections might still be useful:

- **Promotions during leave:** If you are on a leave of absence, you can ask your employer in writing to tell you about job, promotion or training openings you qualify for. Your employer must then let you know about them in writing (s. 209, s. 210.1). This also applies during medical leave (s. 239(5)).
- **Medical leave can't count against you:** Your employer can't hold the fact that you took, or plan to take, medical leave against you when deciding whether to promote or train you (s. 239(6)).
- **Protection from payback (reprisal):** Your employer also can't use the fact that you made a complaint or used your rights under this law against you in promotion or training decisions. I

## Manual review (3 answers read in full)
| Question | Correct / honest | Answers the question | Follows prompt rules |
|---|---|---|---|
| Ministry of Labour complaint (k=5) | ✅ | ✅ federal route + provincial pointer | ✅ |
| Vacation pay if laid off (k=9) | ✅ honest about the gap | ❌ s. 188 not retrieved | ⚠️ ESDC referral not from the sections |
| Raise/promotion trap (k=9) | ✅ says not covered | ✅ helpful tips | ❌ listed citations when not covered |

- Prompt rules followed in 1 of 3 reviewed answers; the slips (outside knowledge, citations on a
  "not covered" answer) both came at k=9, with more context.
- A true/false `covered` can't express "partly answered". A future schema could add that.

## Answer findings
- Scores use the revised test set (see Test Set). The relabelled question was found when reading Claude's answer, which correctly explained the federal complaint route.
- k=5 was the chosen parameter after getting the initial retrieval numbers. After comparing answer quality of k=5 vs k=9 (with Claude answers generated), k=5 is still the choice. This is a tool for legal information, so sticking to the source material is the most important aspect aside from assisting the user. As previously stated, k=9 cites the correct section for one more question (0.83 → 0.89), and answers two more (0.83 → 0.94), with one only partially answered. In the answers I reviewed, both rule breaks came at k=9, including the more serious one: adding information from outside the provided sections. Those slips aren't worth one or two extra answers. 
- **Generation never failed when given the right section:** "cites the right section" equals
  "retrieval hit" at both k. Every remaining miss is a retrieval miss, all from the
  "laid off" → "terminates" vocabulary mismatch found earlier.
- **Honest refusals:** when search missed at k=5, Claude said "not covered" instead of guessing (as k=9, one miss was partly answered; se Manual review);
  both true traps were flagged (1.00 at both k).